In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Embedding output directory. UPDATE for your use case.
emb_output_dir = '/raid/embeddings/BeIR/trec-news-generated-queries/fp32_768d'

# Load huggingface dataset
ds = load_dataset("BeIR/trec-news-generated-queries")
num_rows = ds['train'].num_rows
print(f'Number of rows: {num_rows}')

print(ds['train'].features)

/raid/cuvs-bench-runner/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of rows: 1760922
{'_id': Value('string'), 'title': Value('string'), 'text': Value('string'), 'query': Value('string')}


# Embed Text Passages

In [2]:
# Download from the 🤗 Hub. May need to create a token and "acknowledge license" on HF.
model = SentenceTransformer("google/embeddinggemma-300m") #, token='hf_xxx')

# Start a pool with all available GPUs
pool = model.start_multi_process_pool()

# Encode with batch_size to optimize speed
document_embeddings = model.encode_document(ds['train']['text'], pool=pool,
                                            batch_size=128, show_progress_bar=True)

# Optional: Stop the pool when finished
model.stop_multi_process_pool(pool)

print(document_embeddings.shape)

Chunks: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 353/353 [1:14:10<00:00, 12.61s/it]


(1760922, 768)


In [3]:
# Write data columns as individual numpy files per column in same format
# as processed miracl dataset.

# Create directory and parents if they don't exist
Path(emb_output_dir).mkdir(parents=True, exist_ok=True)

np.save(os.path.join(emb_output_dir, "id.npy"), np.array(ds['train']['_id'], dtype='U36'))
np.save(os.path.join(emb_output_dir, "embedding.npy"), document_embeddings)